In [ ]:
import os
import sqlite3

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import (
    ConsoleSpanExporter,
    SimpleSpanProcessor,
    SpanExporter,
    SpanExportResult,
)

from gitsource import GithubRepositoryDataReader
from minsearch import Index
from rag_helper import RAGBase

load_dotenv("../.env")
client = OpenAI()

In [ ]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

index = Index(text_fields=["content"], keyword_fields=["filename"])
index.fit(documents)

print(f"{len(documents)} pages loaded")

### Q1 — First trace

In [ ]:
provider = TracerProvider()
provider.add_span_processor(SimpleSpanProcessor(ConsoleSpanExporter()))
trace.set_tracer_provider(provider)
tracer = trace.get_tracer("llm-zoomcamp")

In [ ]:
class RAGTraced(RAGBase):

    def search(self, query, num_results=5):
        with tracer.start_as_current_span("search"):
            return super().search(query, num_results)

    def llm(self, prompt):
        with tracer.start_as_current_span("llm") as span:
            response = super().llm(prompt)
            usage = response.usage
            span.set_attribute("input_tokens", usage.prompt_tokens)
            span.set_attribute("output_tokens", usage.completion_tokens)
            cost = usage.prompt_tokens / 1e6 * 0.150 + usage.completion_tokens / 1e6 * 0.600
            span.set_attribute("cost", cost)
            return response

    def rag(self, query):
        with tracer.start_as_current_span("rag"):
            return super().rag(query)

In [ ]:
query = "How does the agentic loop keep calling the model until it stops?"
traced_rag = RAGTraced(index=index, llm_client=client)
answer = traced_rag.rag(query)
print(answer)

### Q2 — Capturing metrics as span attributes

In [ ]:
traced_rag.rag(query)

### Q3 — Span timing

In [ ]:
traced_rag.rag(query)

### Q4 — Saving traces to SQLite

In [ ]:
class SQLiteSpanExporter(SpanExporter):

    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})
            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    attrs.get("cost"),
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self):
        return True

In [ ]:
db_path = "traces.db"

provider2 = TracerProvider()
provider2.add_span_processor(SimpleSpanProcessor(SQLiteSpanExporter(db_path)))
trace.set_tracer_provider(provider2)
tracer = trace.get_tracer("llm-zoomcamp")

traced_rag2 = RAGTraced(index=index, llm_client=client)
traced_rag2.rag(query)

In [ ]:
conn = sqlite3.connect(db_path)
pd.read_sql_query("SELECT DISTINCT name FROM spans", conn)

### Q5 — Querying trace data

In [ ]:
traced_rag2.rag(query)

In [ ]:
df = pd.read_sql_query("SELECT * FROM spans", conn)
df["duration_ns"] = df["end_time"] - df["start_time"]

df[df["name"] != "rag"].groupby("name")["duration_ns"].sum().sort_values(ascending=False)

### Q6 — Token stability across runs

In [ ]:
for _ in range(3):
    traced_rag2.rag(query)

In [ ]:
df2 = pd.read_sql_query("SELECT input_tokens FROM spans WHERE name = 'llm'", conn)
df2